# Identifying Problem Stocks
In this notebook I will identify potential outlier stocks that show abnormal return 
trends with disproportional risk.

Black swan events such as the 2008 financial crisis, COVID-19, the dotcom bubble burst,
GameStop bull run may be legitimate reasons for outlier returns.

However, sometimes financial data may be corrupted for illegitimate reasons including,
double counting, acquisitions & mergers, bankruptcies etc.

It is important to identify potentially corrupted data so that it does not intefere with
algorithm testing and signal generation.

## Imports

In [2]:
import pandas as pd
import numpy as np
import random
from matplotlib import pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import date
from dateutil.relativedelta import relativedelta

from trading_algos import optimization as tao
from trading_algos import datasets as tad
from trading_algos import plots as tap
from trading_algos import utils as tau
from trading_algos.utils import head_tail as ht

%load_ext autoreload
%autoreload 2

2026-06-04 15:53:53.164 | INFO     | trading_algos.config:<module>:11 - PROJ_ROOT path is: /home/jamie/code/JamieW365/trading_algos


## Load Data

In [3]:
# Load a complete collection of S&P500 stocks from repository
# Not interested in survivors here
df_stocks = tad.get_sp500(survivors=False)

In [6]:
df_stocks = df_stocks.loc['2010':'2020']

In [7]:
ht(df_stocks)

Price            Close                                                \
Ticker               A         AA        AAL         AAP        AAPL   
Date                                                                   
2010-01-04   19.856192  35.301846   4.496876   34.742359    6.412383   
2010-01-05   19.640495  34.199326   5.005957   34.535843    6.423470   
2010-01-06   19.570711  35.980328   4.798554   34.836975    6.321296   
2020-12-29  113.111557  21.017693  15.860000  139.215088  131.166779   
2020-12-30  113.265923  21.885479  16.150000  140.076889  130.048386   
2020-12-31  114.327286  21.980843  15.770000  139.943588  129.046646   

Price                                                        ...    Volume  \
Ticker           ABBV        ABNB ABS        ABT       ACGL  ...       XOM   
Date                                                         ...             
2010-01-04        NaN         NaN NaN  18.321970   7.601905  ...  27809100   
2010-01-05        NaN         NaN NaN  18.173939   7.576549  ...  30174700   
2010-01-06        NaN         NaN NaN  18.274876   7.543795  ...  35044700   
2020-12-29  85.467804  150.000000 NaN  98.475838  33.566719  ...  20287700   
2020-12-30  85.933098  148.429993 NaN  98.575844  33.832970  ...  23807300   
2020-12-31  87.467766  146.800003 NaN  99.530312  34.298908  ...  22786500   

Price                                                                       \
Ticker           XRAY      XRX       XYL         XYZ        YUM        ZBH   
Date                                                                         
2010-01-04  1051400.0  5112890       NaN         NaN  2962274.0   805872.0   
2010-01-05   763400.0  3255844       NaN         NaN  3298757.0  1769643.0   
2010-01-06  1595100.0  2634375       NaN         NaN  4178981.0  1315619.0   
2020-12-29   458300.0  1488500  501900.0  15453100.0  1818300.0   757771.0   
2020-12-30   527500.0  1092600  418200.0   9837000.0  1267900.0   440428.0   
2020-12-31   607100.0  1514100  504200.0   6872400.0  1651700.0   514176.0   

Price                                        
Ticker          ZBRA        ZION        ZTS  
Date                                         
2010-01-04  168800.0   3974600.0        NaN  
2010-01-05  168800.0   5605500.0        NaN  
2010-01-06  385300.0  12615200.0        NaN  
2020-12-29  185500.0   1084100.0  1188400.0  
2020-12-30  166100.0    728400.0  1009000.0  
2020-12-31  176500.0    736300.0  1292600.0  

[6 rows x 3300 columns]

In [8]:
df_stocks.shape

(2769, 3300)

In [9]:
print(df_stocks.columns.get_level_values(1).nunique(), 'stocks')

660 stocks


## Identifying Outliers

In [10]:
log_prices = np.log(df_stocks.Close)
ht(log_prices)

Ticker,A,AA,AAL,AAP,AAPL,ABBV,ABNB,ABS,ABT,ACGL,...,XOM,XRAY,XRX,XYL,XYZ,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,2.988516,3.563935,1.503383,3.547960,1.858231,NaN,NaN,NaN,2.908101,2.028399,...,3.627827,3.384530,2.362580,NaN,NaN,2.914239,3.945952,3.355851,2.283970,NaN
2010-01-05,2.977594,3.532206,1.610629,3.541998,1.859959,NaN,NaN,NaN,2.899989,2.025058,...,3.631724,3.372571,2.363738,NaN,NaN,2.910814,3.977117,3.354106,2.318621,NaN
2010-01-06,2.974034,3.582972,1.568315,3.550679,1.843924,NaN,NaN,NaN,2.905527,2.020725,...,3.640330,3.379138,2.354436,NaN,NaN,2.903639,3.976794,3.346389,2.402003,NaN
2020-12-29,4.728375,3.045365,2.763800,4.936020,4.876470,4.448140,5.010635,NaN,4.589811,3.513535,...,3.511756,3.826501,2.816880,4.538675,5.365976,4.597686,4.956380,5.935000,3.577968,5.045283
2020-12-30,4.729738,3.085823,2.781920,4.942191,4.867907,4.453569,5.000113,NaN,4.590826,3.521436,...,3.519720,3.830591,2.826979,4.551356,5.398344,4.598234,4.954870,5.949991,3.588435,5.052308
2020-12-31,4.739065,3.090171,2.758109,4.941239,4.860174,4.471270,4.989071,NaN,4.600462,3.535114,...,3.510544,3.847929,2.840000,4.561329,5.382842,4.589522,4.967080,5.951502,3.593512,5.059768


In [ ]:
# 3 Month Rolling Returns
rolling_log_return = log_prices - log_prices.shift(63)
ht(rolling_log_return)

In [ ]:
df_agg = rolling_log_return.agg({'mean', 'std'}).rename({'mean': 'Return', 'std': 'Risk'}).T
ht(df_agg)

In [ ]:
lq = df_agg.quantile(0.25)
uq = df_agg.quantile(0.75)
iqr = uq - lq
lo = lq - 1.5*iqr
uo = uq + 1.5*iqr

In [ ]:
df_outliers = df_agg[((df_agg > uo) | (df_agg < lo)).any(axis=1)].sort_values(['Risk', 'Return'], ascending=False)
outliers = df_outliers.index.to_list()
df_outliers.shape

In [ ]:
df_outliers.head()

In [ ]:
outliers

In [ ]:
top_ret = abs(df_agg['Return']).sort_values(ascending=False).index[:10]
top_rsk = abs(df_agg['Risk']).sort_values(ascending=False).index[:10]

In [ ]:
df_agg.loc[top_ret, 'Return']

In [ ]:
df_agg.loc[top_rsk, 'Risk']

In [ ]:
outliers

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(12,6), tight_layout=True)

ax[0].set_title('Risk')
ax[0].spines[['left','top','right']].set_visible(False)
sns.boxplot(df_agg[['Risk']], ax=ax[0], orient='h')

ax[1].set_title('Return')
ax[1].spines[['left','top','right']].set_visible(False)
sns.boxplot(df_agg[['Return']], ax=ax[1], orient='h')

In [ ]:
int(np.ceil(len(outliers)/ncols))

In [ ]:
ncols = 5
nrows = int(np.ceil(len(outliers)/ncols))


fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2*nrows), tight_layout=True)

fig.tight_layout(pad=2, rect=[0,0.05,1,0.95])
fig.suptitle(f'Log Price Trend of Outlier Stocks', fontsize=16)

for i, ticker in enumerate(outliers):

    ax = axes[int(i/5), i%5]
    ax.set_title(f"{ticker}")
    ax.set_yticklabels([])
    ax.set_yticks([])
    ax.set_ylabel(' ')
    ax.set_xticklabels([])
    ax.set_xticks([])
    ax.set_xlabel(' ')
    ax.spines[['top','bottom','left','right']].set_visible(False)

    ax.plot((log_prices[ticker]), linewidth=0.5)

# Determine how many empty plots there are on the last row of the figure
num_empty_plots = nrows * ncols - len(outliers)
if num_empty_plots > 0:
    for i in range(1, num_empty_plots + 1):
        # Hide empty plots
        axes[nrows - 1, ncols - i].set_axis_off()

In [ ]:
log_prices

In [15]:
log_returns = (log_prices - log_prices.shift(1))

In [16]:
log_returns

Ticker,A,AA,AAL,AAP,AAPL,ABBV,ABNB,ABS,ABT,ACGL,...,XOM,XRAY,XRX,XYL,XYZ,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,-0.010922,-0.031729,0.107246,-0.005962,0.001728,NaN,NaN,NaN,-0.008112,-0.003341,...,0.003897,-0.011959,0.001158,NaN,NaN,-0.003426,0.031165,-0.001745,0.034651,NaN
2010-01-06,-0.003559,0.050766,-0.042314,0.008682,-0.016034,NaN,NaN,NaN,0.005539,-0.004332,...,0.008606,0.006567,-0.009303,NaN,NaN,-0.007174,-0.000323,-0.007717,0.083382,NaN
2010-01-07,-0.001297,-0.021442,0.029044,-0.000247,-0.001850,NaN,NaN,NaN,0.008250,-0.005900,...,-0.003147,0.013005,0.004662,NaN,NaN,-0.000288,0.022681,-0.025318,0.106160,NaN
2010-01-08,-0.000325,0.024384,-0.019269,0.003945,0.006626,NaN,NaN,NaN,0.005099,-0.001974,...,-0.004020,0.000000,-0.003494,NaN,NaN,0.000288,-0.021228,-0.003256,-0.016319,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-12-24,0.000085,-0.011770,-0.014580,0.008689,0.007683,-0.000193,-0.020266,NaN,0.008341,-0.006291,...,-0.004078,0.004061,-0.008818,0.005201,-0.010458,0.006999,0.003423,-0.007533,-0.003915,0.005428
2020-12-28,0.004423,0.012670,0.025222,-0.011770,0.035141,0.001838,-0.038446,NaN,-0.005182,0.013392,...,0.003360,0.009411,0.030530,0.001196,-0.021878,0.020616,-0.002415,0.015011,0.000462,0.010337
2020-12-29,-0.005105,-0.009033,-0.012531,-0.013376,-0.013404,0.012011,0.006689,NaN,0.004997,-0.000849,...,-0.011324,-0.020472,-0.016021,-0.008605,-0.042719,-0.003010,0.023629,-0.011964,-0.013701,0.004485


In [17]:
log_prices

Ticker,A,AA,AAL,AAP,AAPL,ABBV,ABNB,ABS,ABT,ACGL,...,XOM,XRAY,XRX,XYL,XYZ,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,2.988516,3.563935,1.503383,3.547960,1.858231,NaN,NaN,NaN,2.908101,2.028399,...,3.627827,3.384530,2.362580,NaN,NaN,2.914239,3.945952,3.355851,2.283970,NaN
2010-01-05,2.977594,3.532206,1.610629,3.541998,1.859959,NaN,NaN,NaN,2.899989,2.025058,...,3.631724,3.372571,2.363738,NaN,NaN,2.910814,3.977117,3.354106,2.318621,NaN
2010-01-06,2.974034,3.582972,1.568315,3.550679,1.843924,NaN,NaN,NaN,2.905527,2.020725,...,3.640330,3.379138,2.354436,NaN,NaN,2.903639,3.976794,3.346389,2.402003,NaN
2010-01-07,2.972737,3.561530,1.597358,3.550433,1.842074,NaN,NaN,NaN,2.913777,2.014826,...,3.637183,3.392144,2.359098,NaN,NaN,2.903351,3.999475,3.321071,2.508163,NaN
2010-01-08,2.972412,3.585914,1.578090,3.554377,1.848700,NaN,NaN,NaN,2.918877,2.012851,...,3.633163,3.392144,2.355604,NaN,NaN,2.903639,3.978247,3.317816,2.491844,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-12-24,4.729057,3.041728,2.751110,4.961166,4.854733,4.434291,5.042392,NaN,4.589996,3.500992,...,3.519720,3.837562,2.802371,4.546084,5.430573,4.580081,4.935166,5.931953,3.591207,5.030460
2020-12-28,4.733480,3.054398,2.776332,4.949396,4.889874,4.436129,5.003946,NaN,4.584814,3.514385,...,3.523080,3.846973,2.832901,4.547280,5.408695,4.600697,4.932751,5.946964,3.591668,5.040797
2020-12-29,4.728375,3.045365,2.763800,4.936020,4.876470,4.448140,5.010635,NaN,4.589811,3.513535,...,3.511756,3.826501,2.816880,4.538675,5.365976,4.597686,4.956380,5.935000,3.577968,5.045283


In [ ]:
(abs(log_returns > 1)).mask(log_returns.isna(),np.nan).mean()#.sort_values(ascending=False)

In [ ]:
log_returns['TIE'].loc['2016-04':].head()

#### Identifying Isolated Spikes

In [25]:
isolated_spikes =\
    abs(log_returns > 1).mask(log_returns.isna(), np.nan) &\
   (abs(log_returns.shift(1) < 0.1).mask(log_returns.isna(), np.nan) &
    abs(log_returns.shift(-1) < 0.1).mask(log_returns.isna(), np.nan))

In [31]:
isolated_spikes.sum().sort_values(ascending=False).head(20)

Ticker
CBE     147
TIE      99
CFC      50
MEE      39
BMC       7
CPWR      7
GLK       4
EP        3
PTV       3
RSH       1
PSKY      0
NSC       0
NSM       0
NTAP      0
NTRS      0
NUE       0
NVDA      0
NVR       0
NWL       0
NWS       0
dtype: int64

#### Identifying Spikes with Immediate Reversal

In [51]:
(((log_returns > 1) & (log_returns.shift(1) < -1)) | ((log_returns < 1) & (log_returns.shift(1) > 1))).sum().sort_values(ascending=False).head(20)

Ticker
CBE     223
TIE     151
CFC      85
MEE      64
BMC      10
CPWR      8
GLK       6
PTV       6
EP        3
RSH       2
OGN       0
NSC       0
NSM       0
NTAP      0
NTRS      0
NUE       0
NVDA      0
NVR       0
NWL       0
NWS       0
dtype: int64

#### Identifying Large Residuals

In [54]:
med = log_prices.rolling(21).median()

In [55]:
residual = abs(log_prices - med)

In [60]:
(residual > 2).sum().sort_values(ascending=False).head(20)

Ticker
TIE     241
CBE     222
CFC      84
MEE      60
BMC      18
PTV      11
CPWR     10
NSM       0
NTAP      0
NTRS      0
NUE       0
NVDA      0
NVR       0
NWL       0
NWS       0
NSC       0
NXPI      0
NYT       0
O         0
ODFL      0
dtype: int64